In [1]:
import sys
import numpy as np

sys.path.append("../../../src/")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-04 15:41:02.241300: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-04 15:41:08.731592: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {}
      
    },
  "temp_data_path": "../../../../",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-04 15:41:25,113 [DEBUG] [Rain] Rain is initialized
2023-07-04 15:41:25,115 [DEBUG] [Provisioner] Creating coordinator
2023-07-04 15:41:25,128 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/coord/
2023-07-04 15:41:25,130 [DEBUG] [Coordinator Ambassador] Coordinator Ambassador is initialized
2023-07-04 15:41:25,131 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-04 15:41:25,132 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/coord/
2023-07-04 15:41:25,133 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-04 15:41:25,134 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/
2023-07-04 15:41:25,135 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/
2023-07-04 15:41:25,136 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-04 15:41:25,144 [DEBUG] [Rain] Creating workers
2023-07-04 15:41:25,904 [INFO] [Provisioner] provisioner is serving
2023-07-04 15:41:25,908 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 15:41:25,914 [INFO] [Coordinator] coordinator is serving
2023-07-04 15:41:25,917 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 15:41:25,972 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 15:41:25,978 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 15:41:25,995 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 15:41:25,999 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 15:41:26,005 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 15:41:26,008 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 15:41:26,015 [INFO] [W

157/157 [==============================] - 6s 17ms/step - loss: 0.7042 - accuracy: 0.7793


2023-07-04 15:41:56,752 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-04 15:41:56,754 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider
sending data to divider


2023-07-04 15:41:56,757 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-04 15:41:56,762 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/2_2_trained.pkl from worker2
2023-07-04 15:41:56,767 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-04 15:41:56,769 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/3_3_trained.pkl from worker3


...

157/157 [==============================] - 3s 10ms/step - loss: 0.3626 - accuracy: 0.8924


2023-07-04 15:42:01,559 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 2
2023-07-04 15:42:01,563 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/2_2_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/2_2_trained.pkl from worker2


sending data to divider


2023-07-04 15:42:01,733 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-04 15:42:01,737 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
DEBUG:DeepLearning:Error in calculating the new weights: list index out of range
2023-07-04 15:42:01,765 [DEBUG] [DeepLearning] Iteration 2/3 complete for worker 2.
DEBUG:DeepLearning:Iteration 2/3 complete for worker 2.
2023-07-04 15:42:01,766 [DEBUG] [DeepLearning] Starting iteration 3/3
DEBUG:DeepLearning:Starting iteration 3/3
2023-07-04 15:42:01,768 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
DEBUG:DividerAmbassador:127.0.0.1:50152
2023-07-04 15:42:01,770 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration3 to worker2
DEBUG:DividerAmbassador:divider begins will not send data in iteration3 to worker2
2023-07-04 15:

Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 3ms/step - loss: 0.4490 - accuracy: 0.9205

Test accuracy: 92.0%


In [10]:
# model = rain.train(X_train, y_train, strategy='sync')

In [ ]:
# X_test, y_test = get_test_data()
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

2023-07-04 15:42:19,204 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
INFO:Worker_50151:Running the worker with id: 1 on iteration: 1
2023-07-04 15:42:19,211 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 1
INFO:Worker_50152:Running the worker with id: 2 on iteration: 1
2023-07-04 15:42:19,276 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
INFO:Worker_50152:Running the worker with id: 2 on iteration: 2
2023-07-04 15:42:19,279 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 2
INFO:Worker_50153:Running the worker with id: 3 on iteration: 2
2023-07-04 15:42:19,320 [ERROR] [Worker_50153] Error executing the file: operands could not be broadcast together with shapes (20000,784) (2,) 
ERROR:Worker_50153:Error executing the file: operands could not be broadcast together with shapes (20000,784) (2,) 
2023-07-04 15:42:19,326 [ERROR] [Worker_50153] Error uploading the file: [Errno 2] No such file or directory: '../..

sending data to divider
sending data to divider
sending data to divider


2023-07-04 16:14:19,252 [ERROR] [Worker_50151] Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/X_train_1.npy'
ERROR:Worker_50151:Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/X_train_1.npy'
2023-07-04 16:14:19,279 [ERROR] [Worker_50151] Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/y_train_1.npy'
ERROR:Worker_50151:Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/y_train_1.npy'
2023-07-04 16:14:19,303 [ERROR] [Worker_50151] Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/1.pkl'
ERROR:Worker_50151:Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/1.pkl'
2023-07-04 16:14:19,316 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
INFO:Worker_50151:Running the worker with id: 1 on iteration: 1
20

Error in loading the data:  [Errno 2] No such file or directory: '../../../..//RainData/worker/1.pkl'
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-04 16:19:50,473 [ERROR] [Worker_50151] Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/X_train_1.npy'
ERROR:Worker_50151:Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/X_train_1.npy'
2023-07-04 16:19:50,474 [ERROR] [Worker_50153] Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/X_train_3.npy'
2023-07-04 16:19:50,485 [ERROR] [Worker_50151] Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/y_train_1.npy'
ERROR:Worker_50151:Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/y_train_1.npy'
2023-07-04 16:19:50,523 [ERROR] [Worker_50151] Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/1.pkl'
ERROR:Worker_50151:Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/1.pkl'
2

Error in loading the data:  [Errno 2] No such file or directory: '../../../..//RainData/worker/1.pkl'
Error in receiving the data:  'NoneType' object is not subscriptable
Error in loading the data:  [Errno 2] No such file or directory: '../../../..//RainData/worker/1.pkl'
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-04 16:21:06,951 [ERROR] [Worker_50151] Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/2.pkl'
ERROR:Worker_50151:Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/2.pkl'
2023-07-04 16:21:06,970 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 2
INFO:Worker_50151:Running the worker with id: 1 on iteration: 2
2023-07-04 16:21:06,973 [ERROR] [Worker_50152] Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/2.pkl'
2023-07-04 16:21:06,977 [ERROR] [Worker_50153] Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/2.pkl'
ERROR:Worker_50153:Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/2.pkl'
ERROR:Worker_50152:Error downloading the file: [Errno 2] No such file or directory: '../../../..//RainData/worker/2.pkl'
2023-07-04 16:21:06,992 [ERROR] [W

Error in loading the data:  [Errno 2] No such file or directory: '../../../..//RainData/worker/2.pkl'
Error in receiving the data:  'NoneType' object is not subscriptable
Error in loading the data:  [Errno 2] No such file or directory: '../../../..//RainData/worker/2.pkl'
Error in receiving the data:  'NoneType' object is not subscriptable
Error in loading the data:  [Errno 2] No such file or directory: '../../../..//RainData/worker/2.pkl'
Error in receiving the data:  'NoneType' object is not subscriptable
